# Integrative Analysis of Breast Cancer Molecular Subtypes, Drug Safety and Clinical Trial Trends
**Waihida Yashmin** · M.Sc. Biotechnology · SRM Institute of Science and Technology · Independent project

This notebook presents the results of `run_pipeline.py` in the order of the objectives. Run the pipeline first. The notebook only reads its outputs, so every number here matches the report tables.

In [ ]:
import json, sqlite3
from pathlib import Path
import pandas as pd
from IPython.display import Image, display

ROOT = Path("..")
TAB, FIG = ROOT / "outputs/tables", ROOT / "outputs/figures"
con = sqlite3.connect(ROOT / "data/processed/breast_cancer.db")
QUERIES = [q for q in (ROOT / "sql/analysis_queries.sql").read_text().split(";") if "SELECT" in q]
pd.set_option("display.precision", 3)

## Workflow

In [ ]:
display(Image(ROOT / "docs/methodology_workflow.png", width=850))

## Objective 1 — Data quality and cohort description

In [ ]:
display(pd.read_csv(TAB / "cleaning_report.csv"))
display(pd.read_csv(TAB / "table1_cohort_characteristics.csv"))
for f in ["fig1_subtype_distribution", "fig2_age_tumour_size", "fig3_receptor_status"]:
    display(Image(FIG / f"{f}.png", width=700))

Discordance between clinical HER2 status and the molecular HER2-enriched label (SQL query 4):

In [ ]:
pd.read_sql(QUERIES[3], con)

> **Observation (write your own):** Which subtype is largest and which smallest? How often does IHC/SNP6 status disagree with the PAM50 label, and why might that happen?

## Objective 2 — Clinical and genomic features by subtype
H0: the feature is independent of subtype. Read Cramér's V for strength and the FDR-adjusted q-value for significance.

In [ ]:
display(Image(FIG / "fig4_mutation_heatmap.png", width=650))
display(Image(FIG / "fig5_marker_expression.png", width=650))
chi = pd.read_csv(TAB / "table2_chi_square.csv")
display(chi[["label", "type", "n", "chi2", "dof", "p_permutation", "q_fdr", "cramers_v", "effect"]])
display(pd.read_csv(TAB / "table3_kruskal_wallis.csv"))
display(Image(FIG / "fig6_effect_sizes.png", width=650))

Top mutated genes per subtype, queried directly from the database (SQL query 2):

In [ ]:
pd.read_sql(QUERIES[1], con)

> **Observation:** Is TP53 mostly in Basal/HER2E tumours and PIK3CA/GATA3/MAP3K1/CDH1 mostly in luminal tumours, as published for METABRIC? Do ERBB2/GRB7 and KRT5/KRT17/FOXC1 confirm the subtype labels?

## Objective 3 — Survival by subtype

In [ ]:
display(pd.read_csv(TAB / "table4_survival_by_subtype.csv"))
display(Image(FIG / "fig7_km_curves.png", width=850))
display(pd.read_csv(TAB / "table5_cox_model.csv"))
display(Image(FIG / "fig8_cox_forest.png", width=700))

> **Observation:** Which subtypes carry the highest risk after adjustment? Why might HER2-enriched patients do worse here than they would today (pre-trastuzumab cohort)?

## Objective 4 — Can routine pathology recover the molecular subtype?
Two feature sets compared under class imbalance. Expression is expected to score higher because PAM50 is defined from expression; the gap is the result.

In [ ]:
display(pd.read_csv(TAB / "table6_model_comparison.csv"))
display(Image(FIG / "fig9_model_comparison.png", width=750))
for m in json.loads((TAB / "test_metrics.json").read_text()):
    print(m)
for s in ["clinical_ihc", "gene_expression"]:
    print(s); display(pd.read_csv(TAB / f"table7_classification_report_{s}.csv", index_col=0))
display(Image(FIG / "fig10_confusion_matrices.png", width=850))
display(Image(FIG / "fig11_feature_importance.png", width=850))

> **Observation:** Which subtypes does the clinical model confuse (for example LumA vs LumB, Normal-like)? What routine marker that METABRIC lacks (e.g. Ki-67) might close the gap?

## Objective 5 — Drug eligibility, safety and trial trends

In [ ]:
display(pd.read_sql("SELECT * FROM v_eligibility_by_subtype ORDER BY subtype_code, share_eligible DESC", con))
display(pd.read_csv(TAB / "table8_drug_safety.csv"))
display(Image(FIG / "fig12_serious_share_by_class.png", width=650))
display(Image(FIG / "fig13_trials_over_time.png", width=750))
display(Image(FIG / "fig14_trials_by_drug_class.png", width=750))
pd.read_sql(QUERIES[6], con)

## Objective 6 — Dashboard
Power BI tables are in `powerbi/data/`, and the build guide is `powerbi/DASHBOARD_SPEC.md`. Add screenshots of the three pages to the Results chapter.

In [ ]:
con.close()